# FORESEE Models: U(1)B-L Gauge Boson

## Load Libraries 

In [ ]:
import sys, os
src_path = "../../"
sys.path.append(src_path)
import numpy as np
from src.foresee import Foresee, Utility, Model
from src.utils.utility import BREM_MASSES
from matplotlib import pyplot as plt
import src.foresee as foresee_module

In [ ]:
#Create symlink to direct production spectra.
try: os.unlink('model/direct')
except: pass
os.symlink(
    src=os.path.normpath('../../../files/direct/U(1)B-L'),
    dst='model/direct',
    target_is_directory=True,
)

## 1. Specifying the Model

The phenomenology of the U(1)B-L gauge boson $X$ can be described by the following Lagrangian

\begin{equation}
 \mathcal{L} = \frac{1}{2} \textcolor{red}{m_{X}}^2 X^2  - \textcolor{red}{g_{B-L}} \sum x_{f}\bar f \gamma^\mu f X_\mu
\end{equation}

with the $U(1)_{B-L}$ gauge boson mass $m_{X'}$ and the coupling strength to fermion $g_{B-L}$ as free parameters.The parameters $x_{f}$ specify the baryon minus lepton number of different fermions, in this case $x_{u}$ = $x_{d}$=1/3 and $x_{e}$ = $x_{\nu}$ = -1 for all three generations.  

In [ ]:
energy = "14"
modelname="U(1)B-L"
model = Model(modelname, path="./")

# Builder parameters, matching the build.py / load_model() defaults.
nsample_2body = 2000
generators_light = ["EPOSLHC", "SIBYLL", "QGSJET"][:1]
brem_configurations = ["Brem_QRA_L1.5", "Brem_QRA_L1.0", "Brem_QRA_L2.0"][:1]

**Production:** If the gauge boson is sufficiently light, it is primarily produced in the decay of pseudoscalar mesons $\pi^0$, $\eta$ and $\eta' \to \gamma X$. The branching fractions can be found in Tab. 3 of [1801.04847](https://arxiv.org/pdf/1801.04847.pdf). We find 

\begin{align}
    &\text{BR}(\pi^0 \to X \gamma) = \frac{g_{B-L}^2}{(\varepsilon e)^2}\times\text{BR}(\pi^0\to A'\gamma)\\
    &\text{BR}(\eta \to X \gamma) = \frac{4g_{B-L}^2}{(\varepsilon e)^2} \times\frac{|\mathrm{BW}_\omega(m_X) + \mathrm{BW}_\phi(m_X)|^2}{|\mathrm{BW}_\omega(m_X) + 9\,\mathrm{BW}_\rho(m_X) - 2\,\mathrm{BW}_\phi(m_X)|^2}\times\text{BR}(\eta \to A' \gamma)\\
    &\text{BR}(\eta' \to X \gamma) = \frac{4g_{B-L}^2}{(\varepsilon e)^2} \times\frac{|\mathrm{BW}_\omega(m_X) - 2\,\mathrm{BW}_\phi(m_X)|^2}{|\mathrm{BW}_\omega(m_X) + 9\,\mathrm{BW}_\rho(m_X) + 4\,\mathrm{BW}_\phi(m_X)|^2}\times\text{BR}(\eta' \to A' \gamma)  \ ,
\end{align}

where $BW_{V} = (1-m_{A'}^2/m_V^2 - i \Gamma_V/m_V)^{-1}$.  In the following, we model the production using `EPOSLHC`, `SIBYLL`, `QGSJET`. 

In [ ]:
def BW(self, mass, pid):
    return 1 / (1 - (mass/self.masses(pid))**2 - 1j * self.widths(pid)/self.masses(pid))

foresee_module.BW = BW

In [ ]:
model.add_production_2bodydecay(
    pid0 = "111",
    pid1 = "22",
    br = "2.*0.99  * (coupling/0.303)**2 * (1-mass**2/self.masses(pid0)**2)**3 ",
    generator = generators_light,
    energy = energy,
    nsample = nsample_2body, 
)

model.add_production_2bodydecay(
    pid0 = "221",
    pid1 = "22",
    br = "4.*2.*0.39 * (coupling/0.303)**2 * (1-mass**2/self.masses(pid0)**2)**3 * np.abs( (BW(self,mass,'223') + BW(self,mass,'333')) / (BW(self,mass,'223')  + 9*BW(self,mass,'113')  - 2*BW(self,mass,'333')) )**2",
    generator = generators_light,
    energy = energy,
    nsample = nsample_2body, 
)

model.add_production_2bodydecay(
    pid0 = "331",
    pid1 = "22",
    br = "4.*2.*0.023 * (coupling/0.303)**2 * (1-mass**2/self.masses(pid0)**2)**3 * np.abs( (BW(self,mass,'223') - 2*BW(self,mass,'333')) / (BW(self,mass,'223')  + 9*BW(self,mass,'113')  + 4*BW(self,mass,'333')) )**2",
    generator = generators_light,
    energy = energy,
    nsample = nsample_2body, 
)

Additionally, the $X$ can be produced via vector meson decays $V \to A' P$, where $V = \rho, \rho^0, \omega, \phi$. The branching fractions for these processes are
\begin{align}
    &\text{BR}(\rho \to X\pi) =\frac{4g_{B-L}^2}{(\varepsilon e)^2}\times\text{BR}(\rho\to A'\pi)\\
    &\text{BR}(\omega \to X \eta) = \frac{4g_{B-L}^2}{(\varepsilon e)^2}\times\text{BR}(\omega\to A'\eta)\\
    &\text{BR}(\phi \to X \eta) = \frac{g_{B-L}^2}{(\varepsilon e)^2}\times\text{BR}(\phi\to A'\eta) \ .
\end{align}

In [ ]:
# model.add_production_2bodydecay(
#     pid0 = "213", #rho+
#     pid1 = "211", #pi+
#     br = "4. * (coupling/0.303)**2 * 4.5e-4 * ((self.masses('pid0')**2 - (self.masses('pid1') + mass)**2)*(self.masses('pid0')**2 - (self.masses('pid1') - mass)**2)/(self.masses('pid0')**2 - self.masses('pid1')**2)**2)**(3/2)",
#     generator = generators_light,
#     energy = energy,
#     nsample = nsample_2body, 
# )

# model.add_production_2bodydecay(
#     pid0 = "-213", #rho-
#     pid1 = "-211", #pi-
#     br = "4. * (coupling/0.303)**2 * 4.5e-4 * ((self.masses('pid0')**2 - (self.masses('pid1') + mass)**2)*(self.masses('pid0')**2 - (self.masses('pid1') - mass)**2)/(self.masses('pid0')**2 - self.masses('pid1')**2)**2)**(3/2)",
#     generator = generators_light,
#     energy = energy,
#     nsample = nsample_2body, 
# )

model.add_production_2bodydecay(
    pid0 = "113", #rho0
    pid1 = "111", #pi0
    br = "4. * (coupling/0.303)**2 * 4.7e-4 * ((self.masses('pid0')**2 - (self.masses('pid1') + mass)**2)*(self.masses('pid0')**2 - (self.masses('pid1') - mass)**2)/(self.masses('pid0')**2 - self.masses('pid1')**2)**2)**(3/2)",
    generator = generators_light,
    energy = energy,
    nsample = nsample_2body, 
)


model.add_production_2bodydecay(
    pid0 = "223", #omega
    pid1 = "221", #eta
    br = "4. * (coupling/0.303)**2 * 4.5e-4 * ((self.masses('pid0')**2 - (self.masses('pid1') + mass)**2)*(self.masses('pid0')**2 - (self.masses('pid1') - mass)**2)/(self.masses('pid0')**2 - self.masses('pid1')**2)**2)**(3/2)",
    generator = generators_light,
    energy = energy,
    nsample = nsample_2body, 
)

model.add_production_2bodydecay(
    pid0 = "333", #phi
    pid1 = "221", #eta
    br = "(coupling/0.303)**2 * 1.306e-2 * ((self.masses('pid0')**2 - (self.masses('pid1') + mass)**2)*(self.masses('pid0')**2 - (self.masses('pid1') - mass)**2)/(self.masses('pid0')**2 - self.masses('pid1')**2)**2)**(3/2)",
    generator = generators_light,
    energy = energy,
    nsample = nsample_2body, 
)

The $U(1)_{B-L}$ gauge bosons can also be produced via dark Bremsstrahlung, so coherent radiation off a proton in processes such as $p p \to p p A'$. The spectra for LLPs have been obtained following the description in [1708.09389](https://arxiv.org/abs/1708.09389) and are provided in the `model/direct` directory.


In [ ]:
model.add_production_direct(
    label = "Brem",
    energy = energy,
    configuration = brem_configurations,
    coupling_ref=0.303,
    masses = BREM_MASSES,
)

At high masses, the production of $U(1)_{B-L}$ gauge bosons is dominated by Drell-Yan production, so quark-anti-quark fusion $q q \to X ...$. Here we use the dark photon spectra obtained in [1810.01879](https://arxiv.org/pdf/1810.01879.pdf) using MadGraph plus Pythia, which are provided in the `model/direct` directory. 

In [ ]:
masses_dy = [1.5849, 1.7783, 1.9953, 2.2387, 2.5119, 2.8184, 3.1623, 3.9811,
            5.0119, 6.3096, 7.9433, 10.0, 12.0, 15.0, 17.0, 20.0, 25.0, 30.0, 
            50.0, 70.0, 100.0]
model.add_production_direct(
   label = "DY",
   energy = energy,
   coupling_ref=1,
   masses = masses_dy,
   condition='True',
)

**Decay:** The $U(1)_{B-L}$ gauge boson can decay into a varity of light states. Here we use the lifetime and the decay branching fractions as calculated in [2201.01788](https://arxiv.org/abs/2201.01788) with the DeLiVeR tool. 

In [ ]:
model.set_ctau_1d(
    filename="model/ctau.txt", 
)

# Decay final states as (mode, PDG IDs) pairs (DeLiVeR, arXiv:2201.01788). 
# Each mode loads model/br/bfrac_<mode>.txt, including the charge-specific
# hadronic sub-channels (KK_c, KK_n, 4pi_c, ...).
decay_channels = [
    ("elec", [11, -11]), ("muon", [13, -13]),
    ("nue",  [12, -12]), ("numu", [14, -14]), ("nutau", [16, -16]),
    ("3pi",          [211, -211, 111]),
    ("PiGamma",      [111, 22]),
    ("EtaGamma",     [221, 22]),
    ("EtaOmega",     [221, 223]),
    ("EtaPhi",       [221, 333]),
    ("KKpipi_0",     None),              # K+ pi- K- pi+
    ("KKpipi_1",     None),              # KS pi0 K+- pi-+
    ("KKpipi_2",     None),              # K+- pi0 KS pi-+
    ("KKpipi_3",     None),              # KS pi-+ K-+ pi0
    ("KK_c",         [321, -321]),       # K+ K-
    ("KK_n",         [310, 130]),        # K0bar K0 -> K_S K_L
    ("KKpi_0",       [130, 310, 111]),   # K_L K_S pi0
    ("KKpi_1",       [321, -321, 111]),  # K+ K- pi0
    ("KKpi_2",       [321, -211, 311]),  # K+- pi-+ K0 (representative)
    ("OmPiPi_c",     [223, 211, -211]),  # omega pi+ pi-
    ("OmPiPi_n",     [223, 111, 111]),   # omega pi0 pi0
    ("PhiPiPi_c",    [333, 211, -211]),  # phi pi+ pi-
    ("PhiPiPi_n",    [333, 111, 111]),   # phi pi0 pi0
    ("ppbar",        [2212, -2212]),
    ("nnbar",        [2112, -2112]),
]
decay_modes = [mode for mode, _ in decay_channels]
finalstates = [fs for _, fs in decay_channels]
filenames = ["model/br/" + "bfrac_" + mode + ".txt"
             for mode in decay_modes]

model.set_br_1d(modes=decay_modes, finalstates=finalstates, filenames=filenames)

We can now initiate FORESEE with the model that we just created. 

In [ ]:
foresee = Foresee(path=src_path)
foresee.set_model(model=model)

## 2. Event Generation

In the following, we want to study one specific benchmark point with $m_{X}=30$ MeV and $g_{B-L}=10^{-5}$ and export events as a HEPMC file. 

In [ ]:
mass, coupling, = 0.03, 1e-5

First, we will produce the corresponding flux for this mass and a reference coupling $g_{ref}=1$. 

In [ ]:
%%time
plot=foresee.get_llp_spectrum(mass=mass, coupling=1, do_plot=True)
os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Spectrum_{modelname}.pdf", bbox_inches="tight")
plot.show()

Next, let us define the configuration of the detector (in terms of position, size and luminosity). Here we choose FASER during 2022/2023.  Note that we explicly specify the detectable decay channels, to exclude decays into neutrinos. 

In [ ]:
foresee.set_detector(
    distance=474, 
    selection="np.sqrt(x.x**2 + (x.y+0.065)**2)<.1", 
    length=1.5, 
    luminosity=60, 
    channels=["elec"],
)

For our benchmark point, let us now look at how many particle decay inside the decay volume. We also export 1000 unweighted events as a HEPMC file. 

In [ ]:
setupnames  =     generators_light
modes = {'111':   generators_light, 
         '221':   generators_light, 
         "Brem":  brem_configurations,
}

momenta, weights, _ = foresee.write_events(
    mass = mass, 
    coupling = coupling, 
    energy = energy, 
    numberevent = 1000,
    filename = "model/events/test.hepmc", 
    return_data = True,
    weightnames=setupnames,
    modes=modes,
)

for isetup, setup in enumerate(setupnames):
    print("Expected number of events for "+setup+":", round(sum(weights[:,isetup]),3))

Let us plot the resulting energy distribution

In [ ]:
fig = plt.figure(figsize=(7,5))
ax = plt.subplot(1,1,1)
energies = [p.e for p in momenta], 
for isetup, setup in enumerate(setupnames):
    ax.hist(energies, weights=weights[:,isetup], bins=np.logspace(2,4, 20+1), histtype='step', label=setup) 
ax.set_xscale("log")
ax.set_xlim(1e2,1e4) 
ax.set_xlabel("E [GeV]") 
ax.set_ylabel("Number of Events per Bin") 
ax.legend(frameon=False, labelspacing=0, fontsize=14, loc='upper left')
plt.savefig(f"figures/{modelname}/E_distribution_{modelname}.pdf", bbox_inches="tight")
plt.show()

## 3. Sensitivity Reach

In the following, we will obtain the projected sensitivity for the LLP model. For this, we first define a grid of couplings and masses, and then produce the corresponding fluxes. 

In [ ]:
masses = [m for m in BREM_MASSES if 0.01 <= m <= 2.0]
# Extra points around each production channel kinematic endpoint, from
# utility.production_thresholds(model, mass_range=(0.01, 2.05)).
thresholds = [0.120,
    0.13093, 0.13498, 0.13903, 0.22058, 0.2274, 0.22775, 0.23422, 0.2348,
    0.24184, 0.45745, 0.4716, 0.48575, 0.53143, 0.54786, 0.5643, 0.62466,
    0.64398, 0.6633, 0.92905, 0.95778, 0.98651,
]
masses = sorted(masses + thresholds)
couplings = np.logspace(-8,-3,121)

# Use cached LLP spectra: get_llp_spectrum recomputes on every call, 
# so skip any masses already saved in model/LLP_spectra/.
for mass in masses:
    if not os.path.exists(f"model/LLP_spectra/{energy}TeV_m_{mass}.txt.gz"):
        foresee.get_llp_spectrum(mass=mass, coupling=1)

We can now plot the `production rate vs mass` using the `foresee.plot_production()` function.

In [ ]:
import matplotlib.colors as mcolors

colors= list(mcolors.TABLEAU_COLORS.keys())

productions=[
    {"channels": ["111"] , "color": colors[0]      , "label": r"$\pi^0 \to \gamma A'$", "generators": generators_light },
    {"channels": ["221"] , "color": colors[1]   , "label": r"$\eta \to \gamma A'$" , "generators": generators_light  },
    {"channels": ["331"] , "color": colors[2]   , "label": r"$\eta' \to \gamma A'$" , "generators": generators_light  },
    {"channels": ["113"] , "color": colors[3]      , "label": r"$\rho^0 \to \pi^0 A'$", "generators": generators_light  },
    {"channels": ["223"] , "color": colors[5]      , "label": r"$\omega \to \eta A'$", "generators": generators_light  },
    {"channels": ["333"] , "color": colors[7]      , "label": r"$\phi \to \eta A'$", "generators": generators_light  },
    {"channels": ["Brem"], "color": colors[8], "label": r"Bremsstrahlung"       , "generators": brem_configurations},
    {"channels": ["DY"], "color": colors[9], "label": r"Drell-Yan"       , "generators":["DY"]},
]

branchings = [
    ["elec"     , "red"        , "solid" , r"$e^+e^-$"                , 0.020, 0.48],
    ["muon"     , "orange"     , "solid" , r"$\mu^+\mu^-$"              , 0.145, 0.10],
    ["nue"      , "blue"       , "solid" , r"$\overline{\nu}_\alpha\nu_\alpha$" , 0.020, 0.12],
    ["PiGamma"  , "dodgerblue" , "solid" , r"$\pi^0\gamma$"       , 0.55, 0.02],
    ["3pi"      , "brown"       , "solid" , r"$\pi^0\pi^+\pi^-$"         , 0.45, 0.7],
    ["EtaOmega" , "purple"     , "solid" , r"$\eta\omega$"         , 1.2, 0.03],
    ["OmPiPi_c" , "magenta"    , "solid" , r"$\omega\pi^+\pi^-$"    , 1.07, 0.05],
    ["KK_n"     , "darkgreen"  , "solid" , r"$K_S K_L$"           , 1.1, 0.3],
    ["KK_c"     , "green"      , "solid" , r"$K^+K^-$"            , 1.1, 0.45],
]

plot, ax, ax2 = foresee.plot_production(
    masses = masses,
    productions = productions,
    energy=energy,
    condition="logth<-3.7 and logp>2",
    xlims=[0.01, 2.0],ylims=[2e4, 4e13],
    xlabel=r"Mass [GeV]",
    ylabel=r"Production Rate $\sigma/g_{B-L}^2$ [pb]",
    title=r"$\theta < 0.2$ mrad and $E > 100$ GeV",
    legendloc=(1.01,1),
    fs_label=12,
    fs_label_br=9,
    ncol=3,
    branchings=branchings,
    figsize=(7,6)
)

# rug: one tick per grid mass, pinned to the bottom axis edge
# from matplotlib.transforms import blended_transform_factory
# trans = blended_transform_factory(ax.transData, ax.transAxes)
# ax.plot(masses, [0]*len(masses), marker="|", linestyle="none",
#         color="black", markersize=10, markeredgewidth=0.8,
#         transform=trans, clip_on=False, zorder=5)

os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Production_{modelname}.pdf", bbox_inches="tight")
plot.show()

Let us now scan over various masses and couplings, and record the resulting number of events. Note that here we again consider the FASER configuration, which we set up before.

In [ ]:
setupnames = ['EPOSLHC']
modes = None

if energy == "13.6": detectors = [["FASER_R3"  , 480, "np.sqrt(x.x**2 + x.y**2)< .1", 1.5, 250 ,  None]]
elif energy == "14": detectors = [["FASER_HL"  , 480, "np.sqrt(x.x**2 + x.y**2)< .1", 1.5, 3000,  None], 
                                  ["FASER2_HL" , 650, "-1.5<x.x<1.5 and -.5<x.y<.5" , 10 , 3000,  None]]

condition = f"np.sqrt(p**2 + mass**2) > 100"

for detector in detectors: 

    #setup detector
    dlabel, distance, selection, length, luminosity, channels  = detector

    #skip detectors already precomputed (the plot cell reads them); scan only missing ones
    if all(os.path.exists(f"model/results/{energy}TeV_{dlabel}_{label}.npy") for label in setupnames):
        continue

    foresee.set_detector(distance=distance, selection=selection, length=length, luminosity=luminosity, channels=channels)

    #get reach  
    list_nevents = {label:[] for label in setupnames}
    for mass in masses:
        couplings, _, nevents, _, _  = foresee.get_events(mass=mass, energy=energy, couplings = couplings,modes=modes,nsample=10, preselectioncuts = condition)
        for i,label in enumerate(setupnames): list_nevents[label].append(nevents.T[i])  
            
    #save results
    configuration=dlabel
    for label in setupnames: 
        result = np.array([masses,couplings,list_nevents[label]], dtype='object')
        np.save("model/results/"+energy+"TeV_"+configuration+"_"+label+".npy",result)

We can now plot the results. For this, we first specify all detector setups for which we want to show result (filename in model/results directory, label, color, linestyle, opacity alpha for filled contours, required number of events).

In [ ]:
setups = [ 
    ["13.6TeV_FASER_R3_EPOSLHC.npy"   , r"FASER (Run 3)"    , "firebrick"         ,  "solid"  , 0., 3],
    ["14TeV_FASER_HL_EPOSLHC.npy"   , r"FASER (HL-LHC)"    , "red"         ,  "dashed"  , 0., 3],
    ["14TeV_FASER2_HL_EPOSLHC.npy"   , r"FASER2 (HL-LHC)"    , "salmon"         ,  "dashed"  , 0., 3],    
]

Then we specify all the existing bounds, separating the bounds obtained by experimental collaboratios and theory recasts.

In [ ]:
bounds = [ 
    ["bounds_NA64_inv_new.txt", "NA64 inv.",7.6e-2, 5.0e-5, 0], 
    ["bounds_CHARM-II.txt",     "Charm II" ,1.1e-1, 1.1e-4, 20],
    ["bounds_Texono.txt",       "Texono"   ,5.0e-2, 1.2e-4, 0],
    ["bounds_LHCb_prompt.txt",  "LHCb"     ,2.3e-1, 8.0e-5, 0],
    ["bounds_BaBar.txt",        "BaBar"    ,3.0e-1, 4.0e-4, 0],
    ["bounds_NA48.txt",         "NA48"     ,2.5e-2, 7.0e-4, 0],
    ["bounds_KLOE_combined.txt","KLOE"     ,5.4e-1, 7.0e-4, 0], 
    ["bounds_A1.txt",           "A1"       ,1.2e-1, 6.5e-4, 0], 
    ["bounds_E137.txt",         "E137"     ,1.5e-1, 3.0e-8, 0],
    ["bounds_NuCal.txt",        "NuCal"    ,1.9e-1, 2.0e-7, 0],
    ["bounds_Orsay.txt",        "Orsay"    ,4.1e-2, 1e-6, -25],
    ["bounds_CHARM.txt",     "Charm" ,4e-2, 1.0e-7, -5],
    ["bounds_NA64.txt",         "NA64 ee"  ,1.5e-2, 1.2e-4, 0],
    ["bounds_FASER60ifb.txt", r"FASER $(60\text{fb}^{-1})$",2.2e-2, 4e-6, -25], 
    ["bounds_FASER27ifb.txt", r"FASER $(27\text{fb}^{-1})$",1.3e-2, 4e-6, -25], 
    ["bounds_E141.txt",         "E141"     ,1.05e-2, 2.1e-5, 90],  
]


We then specify other projected sensitivitities (filename in model/bounds directory, color, label, label position x, label position y, label rotation)

In [ ]:
projections = [
#     ["limits_SHIP.txt",       "blue",  " SHiP"       , 0.6e+0, 3.0e-7, 0],
#     ["limits_Belle-IIyvv.txt","violet","Belle-II inv", 0.4e+0, 1.5e-5, 0],
#     ["limits_NA_64mu.txt",    "brown", "NA64$\mu$"   , 0.4e+0, 4.0e-5, 0],
#     ["limits_Belle_II.txt",   "navy",  "Belle-II"    , 4.0e-1, 7.0e-5, 0],
]

Finally, we can plot everything using `foresee.plot_reach()`. It returns a matplotlib instance, to which we can add further lines and which we can show or save. 

In [ ]:
plot = foresee.plot_reach(
    setups=setups,
    bounds=bounds,
    projections=projections,
    title = "B-L Gauge Boson", 
    xlims = [0.01,1.5], 
    ylims = [8e-9,1e-3],
    xlabel=r"Gauge boson mass $m_{X}$ [GeV]", 
    ylabel=r"$g_{(B-L)}$" ,
    legendloc=(1.00,0.75),
    linewidths=2,
)

os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Reach_{modelname}.pdf", bbox_inches="tight")
plot.show()